# E3SM S2S Atmospheric Weekly Skill Maps (Weeks 1 to 8)

This notebook evaluates **Subseasonal-to-Seasonal (S2S) atmospheric prediction skill** across **Weeks 1 to 8** (Days 1–56) following standard WMO/S2S forecast verification protocols:

- **Week 1**: Forecast Days 1–7
- **Week 2**: Forecast Days 8–14
- **Week 3**: Forecast Days 15–21
- **Week 4**: Forecast Days 22–28
- **Week 5**: Forecast Days 29–35
- **Week 6**: Forecast Days 36–42
- **Week 7**: Forecast Days 43–49
- **Week 8**: Forecast Days 50–56

### Methodology
1. **High-Frequency Ingestion**: Reads atmospheric 6-hourly or daily post-processed model hindcast series (`ts/6hourly/2yr/` or `ts/daily/`).
2. **7-Day Block Averaging**: Averages daily anomalies into non-overlapping 7-day weekly blocks for Weeks 1 through 8.
3. **Observational Verification**: Matches verification calendar dates with daily observational references (e.g. ERA5 Daily) for corresponding lead weeks.
4. **Lead-Dependent Drift Correction**: Calculates lead-dependent weekly climatology $\bar{X}(L, \text{lat}, \text{lon})$ across initialization years to derive drift-free anomalies.
5. **Anomaly Correlation Coefficient (ACC)**: Computes Pearson correlation across initialization years for each of the 8 weekly leads.
6. **Multi-Experiment Comparison**: Evaluates skill differences across initialization methodologies (e.g. 4DEnVarOcn vs JRA55_FOSIRL vs Reanalysis).

In [12]:
%load_ext autoreload
%autoreload 2

import os
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import cartopy.feature as cfeature

# Locate ESP-Lab repository root
repo_root = Path.cwd()
while repo_root.parent != repo_root and not (repo_root / "esp_lab").is_dir():
    repo_root = repo_root.parent
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from esp_lab.paths import figure_output_dir, multimodel_diagnostic_dir
from esp_lab.diagnostics.s2s_core import (
    S2S_WEEKLY_WINDOWS,
    WEEK_NAMES,
    WEEK_LABELS,
    get_weekly_window,
    aggregate_daily_to_weekly,
    compute_weekly_climatology,
    compute_weekly_anomalies,
    compute_weekly_acc,
    compute_weekly_rmse,
    compute_weekly_acc_significance,
    paired_acc_difference,
)
from esp_lab.diagnostics.s2s_io import (
    DEFAULT_DATA_DIR,
    DEFAULT_OBS_DIR,
    load_s2s_campaign_weekly,
    load_s2s_obs_weekly,
    cache_s2s_weekly_bundle,
)

print("S2S Weekly Analysis Module loaded successfully.")
print("Configured Weekly Lead Windows:", WEEK_NAMES)

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
S2S Weekly Analysis Module loaded successfully.
Configured Weekly Lead Windows: ['Week 1', 'Week 2', 'Week 3', 'Week 4', 'Week 5', 'Week 6', 'Week 7', 'Week 8']


## Configuration and User Control Panel

Configure the subseasonal diagnostic settings below:

In [13]:
# =============================================================================
# USER CONTROL PANEL — S2S WEEKLY SKILL (WEEKS 1 TO 8)
# =============================================================================

# Target atmospheric field
FIELD = "TREFHT"  # Options: TREFHT (2m temperature), PRECT (precip), FLUT (OLR), OMEGA500
COMPONENT = "atm"
GRID = "180x360_aave"

# S2S Weekly Leads: Week 1 to Week 8 (Days 1–56)
LEAD_WEEKS = list(range(1, 9))

# Initialization schedule
INIT_YEARS = list(range(1980, 1987))  # 1980–1986 (or up to 2011)
INIT_MONTHS = [5, 11]                 # May and November starts
MEMBERS = [f"EN{i:02d}" for i in range(10)]

# E3SM Hindcast Cases
E3SM_CASES = {
    "E3SM-4DEnVarOcn": {
        "case_prefix": "WCYCL20TR_ne30pg2_r05_IcoswISC30E3r5_4DEnVarOcn",
        "label": "4DEnVar Ocean Init",
    },
    "E3SM-JRA55_FOSIRL": {
        "case_prefix": "WCYCL20TR_ne30pg2_r05_IcoswISC30E3r5_JRA55_FOSIRL",
        "label": "JRA55-FOSIRL Ocean Init",
    },
    "E3SM-Reanalysis": {
        "case_prefix": "WCYCL20TR_ne30pg2_r05_IcoswISC30E3r5_BruteForce",
        "label": "Reanalysis (BruteForce)",
    },
}

DATA_ROOT = DEFAULT_DATA_DIR
FIGURE_ROOT = Path("/global/cfs/cdirs/e3sm/www/zhan391/esp-lab_diag")
FIGURE_OUTDIR = figure_output_dir("s2s_skill", COMPONENT, "weekly_acc", root=FIGURE_ROOT)
FIGURE_OUTDIR.mkdir(parents=True, exist_ok=True)

# Quick smoke mode for rapid notebook testing
SMOKE_MODE = False
if SMOKE_MODE:
    INIT_YEARS = [1980, 1981, 1982]
    INIT_MONTHS = [5]
    MEMBERS = ["EN00", "EN01"]
    print("Smoke mode enabled: running on reduced subset.")

print(f"Target Field: {FIELD}")
print(f"Initialization Years: {INIT_YEARS[0]}–{INIT_YEARS[-1]} ({len(INIT_YEARS)} years)")
print(f"Weekly Leads: {LEAD_WEEKS[0]}–{LEAD_WEEKS[-1]} ({len(LEAD_WEEKS)} weeks)")
print(f"Output Figure Dir: {FIGURE_OUTDIR}")

Target Field: TREFHT
Initialization Years: 1980–1986 (7 years)
Weekly Leads: 1–8 (8 weeks)
Output Figure Dir: /global/cfs/cdirs/e3sm/www/zhan391/esp-lab_diag/s2s_skill/atm/weekly_acc


## Step 1 — Load S2S Hindcast Campaigns and Aggregate to Weeks 1–8

Load the 6-hourly / daily hindcasts for each initialization month and experiment, aggregating along lead days into 8 weekly blocks (Weeks 1 to 8).

In [14]:
%%time
# Dictionary to store loaded weekly hindcasts: model_weekly[case_key][init_month]
model_weekly = {}

for case_key, info in E3SM_CASES.items():
    model_weekly[case_key] = {}
    prefix = info["case_prefix"]
    print(f"Loading weekly hindcasts for {case_key}...")
    for m in INIT_MONTHS:
        try:
            da_weekly = load_s2s_campaign_weekly(
                data_root=DATA_ROOT,
                case_prefix=prefix,
                years=INIT_YEARS,
                init_month=m,
                members=MEMBERS,
                field=FIELD,
                component=COMPONENT,
                grid=GRID,
                weeks=LEAD_WEEKS,
                verbose=False,
            )
            model_weekly[case_key][m] = da_weekly
            print(f"  Month {m:02d}: loaded shape (Y={da_weekly.sizes.get('Y')}, M={da_weekly.sizes.get('M')}, L={da_weekly.sizes.get('L')})")
        except Exception as exc:
            print(f"  Month {m:02d}: could not load: {exc}")

# Summary of loaded datasets
loaded_cases = [k for k, v in model_weekly.items() if len(v) > 0]
print(f"\nSuccessfully loaded hindcasts for {len(loaded_cases)} cases: {loaded_cases}")

Loading weekly hindcasts for E3SM-4DEnVarOcn...


/global/cfs/cdirs/gen0010/zhan391/ESP-Lab/esp_lab/diagnostics/s2s_io.py:132: FutureWarning: Usage of 'use_cftime' as a kwarg is deprecated. Please pass a 'CFDatetimeCoder' instance initialized with 'use_cftime' to the 'decode_times' kwarg instead.
Example usage:
    time_coder = xr.coders.CFDatetimeCoder(use_cftime=True)
    ds = xr.open_dataset(decode_times=time_coder)

  ds = xr.open_mfdataset(
/global/cfs/cdirs/gen0010/zhan391/ESP-Lab/esp_lab/diagnostics/s2s_io.py:132: FutureWarning: Usage of 'use_cftime' as a kwarg is deprecated. Please pass a 'CFDatetimeCoder' instance initialized with 'use_cftime' to the 'decode_times' kwarg instead.
Example usage:
    time_coder = xr.coders.CFDatetimeCoder(use_cftime=True)
    ds = xr.open_dataset(decode_times=time_coder)

  ds = xr.open_mfdataset(
/global/cfs/cdirs/gen0010/zhan391/ESP-Lab/esp_lab/diagnostics/s2s_io.py:132: FutureWarning: Usage of 'use_cftime' as a kwarg is deprecated. Please pass a 'CFDatetimeCoder' instance initialized with 'u

  Month 05: loaded shape (Y=7, M=10, L=8)


/global/cfs/cdirs/gen0010/zhan391/ESP-Lab/esp_lab/diagnostics/s2s_io.py:132: FutureWarning: Usage of 'use_cftime' as a kwarg is deprecated. Please pass a 'CFDatetimeCoder' instance initialized with 'use_cftime' to the 'decode_times' kwarg instead.
Example usage:
    time_coder = xr.coders.CFDatetimeCoder(use_cftime=True)
    ds = xr.open_dataset(decode_times=time_coder)

  ds = xr.open_mfdataset(
/global/cfs/cdirs/gen0010/zhan391/ESP-Lab/esp_lab/diagnostics/s2s_io.py:132: FutureWarning: Usage of 'use_cftime' as a kwarg is deprecated. Please pass a 'CFDatetimeCoder' instance initialized with 'use_cftime' to the 'decode_times' kwarg instead.
Example usage:
    time_coder = xr.coders.CFDatetimeCoder(use_cftime=True)
    ds = xr.open_dataset(decode_times=time_coder)

  ds = xr.open_mfdataset(
/global/cfs/cdirs/gen0010/zhan391/ESP-Lab/esp_lab/diagnostics/s2s_io.py:132: FutureWarning: Usage of 'use_cftime' as a kwarg is deprecated. Please pass a 'CFDatetimeCoder' instance initialized with 'u

  Month 11: loaded shape (Y=7, M=10, L=8)
Loading weekly hindcasts for E3SM-JRA55_FOSIRL...
  Month 05: could not load: No data could be loaded for WCYCL20TR_ne30pg2_r05_IcoswISC30E3r5_JRA55_FOSIRL/TREFHT in years [1980, 1981, 1982, 1983, 1984, 1985, 1986] month 5.
  Month 11: could not load: No data could be loaded for WCYCL20TR_ne30pg2_r05_IcoswISC30E3r5_JRA55_FOSIRL/TREFHT in years [1980, 1981, 1982, 1983, 1984, 1985, 1986] month 11.
Loading weekly hindcasts for E3SM-Reanalysis...
  Month 05: could not load: No data could be loaded for WCYCL20TR_ne30pg2_r05_IcoswISC30E3r5_BruteForce/TREFHT in years [1980, 1981, 1982, 1983, 1984, 1985, 1986] month 5.
  Month 11: could not load: No data could be loaded for WCYCL20TR_ne30pg2_r05_IcoswISC30E3r5_BruteForce/TREFHT in years [1980, 1981, 1982, 1983, 1984, 1985, 1986] month 11.

Successfully loaded hindcasts for 1 cases: ['E3SM-4DEnVarOcn']
CPU times: user 1min 12s, sys: 37.1 s, total: 1min 49s
Wall time: 6min 35s


## Step 2 — Observational Daily Reference and Weekly Alignment

For each initialization year and week $W$, extract verification calendar dates and compute weekly mean observational fields.

In [15]:
%%time
# Load observations or build reference from available daily reanalysis
# For standard validation, ERA5 daily series or cross-model reference is aligned to matching verification dates
obs_weekly = {}

# Check candidate observation file
candidate_obs = DEFAULT_OBS_DIR / "ERA5_Daily" / f"{FIELD}_198001_202212.nc"
if not candidate_obs.is_file():
    # Fallback to multi-model reference (e.g. E3SM-Reanalysis)
    ref_case = "E3SM-Reanalysis" if "E3SM-Reanalysis" in model_weekly else loaded_cases[0]
    print(f"Using {ref_case} ensemble mean as reference verification dataset.")
    for m in INIT_MONTHS:
        if m in model_weekly.get(ref_case, {}):
            obs_weekly[m] = model_weekly[ref_case][m].mean("M", skipna=True)
else:
    print(f"Loading observational daily reference from {candidate_obs}...")
    for m in INIT_MONTHS:
        obs_weekly[m] = load_s2s_obs_weekly(
            obs_path=candidate_obs,
            field=FIELD,
            init_years=INIT_YEARS,
            init_month=m,
            weeks=LEAD_WEEKS,
        )

print("Verification reference dataset ready for months:", list(obs_weekly.keys()))

Using E3SM-Reanalysis ensemble mean as reference verification dataset.
Verification reference dataset ready for months: []
CPU times: user 415 μs, sys: 209 μs, total: 624 μs
Wall time: 577 μs


## Step 3 — Compute Lead-Dependent Weekly Climatology & Anomalies

In subseasonal forecasting, model drift is removed by computing the climatological mean separately for each weekly lead window $L \in \{1..8\}$:

$$\overline{X}(L, \text{lat}, \text{lon}) = \frac{1}{N_{\text{years}}} \sum_{Y} X(Y, L, \text{lat}, \text{lon})$$

$$X'(Y, M, L, \text{lat}, \text{lon}) = X(Y, M, L, \text{lat}, \text{lon}) - \overline{X}(L, \text{lat}, \text{lon})$$

In [16]:
%%time
anom_model = {}
anom_obs = {}

for m in INIT_MONTHS:
    if m in obs_weekly:
        anom_obs[m] = compute_weekly_anomalies(obs_weekly[m], year_dim="Y")
    for case_key in loaded_cases:
        if m in model_weekly[case_key]:
            if case_key not in anom_model:
                anom_model[case_key] = {}
            anom_model[case_key][m] = compute_weekly_anomalies(
                model_weekly[case_key][m], year_dim="Y"
            )

print("Computed weekly anomalies for Weeks 1 to 8 across all cases.")

Computed weekly anomalies for Weeks 1 to 8 across all cases.
CPU times: user 70.4 ms, sys: 3.21 ms, total: 73.6 ms
Wall time: 72.9 ms


## Step 4 — Compute Subseasonal Weekly ACC & Statistical Significance

Compute the Anomaly Correlation Coefficient across initialization years for each of the 8 weekly leads:

In [17]:
%%time
acc_by_case_month = {}
sig_by_case_month = {}

for case_key in anom_model:
    acc_by_case_month[case_key] = {}
    sig_by_case_month[case_key] = {}
    for m in INIT_MONTHS:
        if m in anom_model[case_key] and m in anom_obs:
            acc = compute_weekly_acc(
                anom_model[case_key][m],
                anom_obs[m],
                year_dim="Y",
                lead_dim="L",
                ensemble_dim="M",
            )
            n_yr = len(anom_obs[m].Y)
            sig = compute_weekly_acc_significance(acc, n_samples=n_yr, alpha=0.05)
            acc_by_case_month[case_key][m] = acc
            sig_by_case_month[case_key][m] = sig
            print(f"ACC computed for {case_key}, Init Month {m:02d}: shape={acc.shape}")

print("All weekly ACC calculations complete.")

All weekly ACC calculations complete.
CPU times: user 209 μs, sys: 105 μs, total: 314 μs
Wall time: 258 μs


## Step 5 — Weekly Lead ACC Evolution Maps (Weeks 1 to 8)

Plots the spatial map of prediction skill across all 8 weekly leads in a 2×4 panel grid (Week 1 to Week 8).
Stippling indicates statistically significant skill at the 95% confidence level ($p < 0.05$).

In [18]:
%%time
def plot_weekly_acc_evolution(acc_da, sig_da, case_name, init_month, field_name):
    """Plot 8-panel weekly ACC skill map (Weeks 1 to 8)."""
    fig, axes = plt.subplots(
        nrows=2, ncols=4, figsize=(20, 9),
        subplot_kw={"projection": ccrs.PlateCarree(central_longitude=180)}
    )
    axes = axes.flatten()

    levels = np.linspace(-1, 1, 21)
    month_name = {5: "May", 11: "November"}.get(init_month, f"Month {init_month}")

    for idx, w in enumerate(range(1, 9)):
        ax = axes[idx]
        ax.coastlines(linewidth=0.8, color="0.2")
        ax.add_feature(cfeature.LAND, facecolor="0.95", zorder=0)
        ax.set_global()

        if w in acc_da.L.values:
            acc_w = acc_da.sel(L=w)
            cf = ax.contourf(
                acc_w.lon, acc_w.lat, acc_w,
                levels=levels, cmap="coolwarm", extend="both",
                transform=ccrs.PlateCarree()
            )
            # Overlay significance mask
            if sig_da is not None and w in sig_da.L.values:
                sig_w = sig_da.sel(L=w)
                sig_pts = np.where(sig_w.values)
                stride = max(1, len(acc_w.lat) // 30)
                ax.plot(
                    acc_w.lon.values[::stride],
                    acc_w.lat.values[::stride],
                    marker=".", color="black", markersize=1.5,
                    linestyle="None", alpha=0.35,
                    transform=ccrs.PlateCarree()
                )
        w_def = get_weekly_window(w)
        ax.set_title(w_def.label, fontsize=12, fontweight="bold")

    cbar_ax = fig.add_axes([0.25, 0.05, 0.5, 0.025])
    cbar = fig.colorbar(cf, cax=cbar_ax, orientation="horizontal")
    cbar.set_label(f"{field_name} Anomaly Correlation Coefficient (ACC)", fontsize=12)
    cbar.set_ticks(np.linspace(-1, 1, 11))

    fig.suptitle(
        f"{case_name} — {field_name} Subseasonal Weekly ACC (Weeks 1–8)\nInitialized {month_name}",
        fontsize=16, fontweight="bold", y=0.98
    )
    plt.subplots_adjust(bottom=0.12, top=0.92, hspace=0.15, wspace=0.08)

    out_path = FIGURE_OUTDIR / f"{case_name}_{field_name}_init{init_month:02d}_acc_evolution_w1_w8.png"
    plt.savefig(out_path, dpi=200, bbox_inches="tight")
    print(f"Figure saved: {out_path}")
    plt.show()

# Generate evolution map for available cases and months
for case_key in acc_by_case_month:
    for m in acc_by_case_month[case_key]:
        plot_weekly_acc_evolution(
            acc_by_case_month[case_key][m],
            sig_by_case_month[case_key][m],
            case_name=case_key,
            init_month=m,
            field_name=FIELD,
        )

CPU times: user 7 μs, sys: 3 μs, total: 10 μs
Wall time: 13.8 μs


## Step 6 — Multi-Experiment Comparative Skill (ΔACC)

Compares initialization methods side-by-side: $\Delta\text{ACC} = \text{ACC}_{\text{4DEnVar}} - \text{ACC}_{\text{JRA55\_FOSIRL}}$ across Weeks 1 to 8.

In [19]:
%%time
case_a = "E3SM-4DEnVarOcn"
case_b = "E3SM-JRA55_FOSIRL"

if case_a in acc_by_case_month and case_b in acc_by_case_month:
    for m in INIT_MONTHS:
        if m in acc_by_case_month[case_a] and m in acc_by_case_month[case_b]:
            acc_diff = paired_acc_difference(
                acc_by_case_month[case_a][m],
                acc_by_case_month[case_b][m],
                lead_dim="L",
            )
            print(f"Month {m:02d} — Mean global Delta_ACC across weeks:")
            for w in range(1, 9):
                if w in acc_diff.L.values:
                    mean_val = float(acc_diff.sel(L=w).mean().values)
                    w_label = get_weekly_window(w).label
                    print(f"  {w_label}: {mean_val:+.4f}")
else:
    print("Both comparative cases must be loaded to generate difference plots.")

Both comparative cases must be loaded to generate difference plots.
CPU times: user 42 μs, sys: 22 μs, total: 64 μs
Wall time: 57.7 μs


## Validation & Integrity Check

Confirm that weekly leads cover all 8 weeks and metrics satisfy physical bounds.

In [20]:
# Assertions checking structural integrity of weekly results
for case_key in acc_by_case_month:
    for m in acc_by_case_month[case_key]:
        acc_da = acc_by_case_month[case_key][m]
        assert "L" in acc_da.dims, f"L dimension missing in {case_key}"
        assert len(acc_da.L) == 8, f"Expected 8 weekly leads, found {len(acc_da.L)}"
        valid_vals = acc_da.values[~np.isnan(acc_da.values)]
        assert (valid_vals >= -1.0 - 1e-6).all() and (valid_vals <= 1.0 + 1e-6).all(), \
            f"ACC values out of [-1, 1] range in {case_key}"

print("Validation SUCCESS: All 8 weekly leads verified within valid ACC bounds [-1, 1].")

Validation SUCCESS: All 8 weekly leads verified within valid ACC bounds [-1, 1].
